# AMI Subset Diarization Pipeline (pyannote.audio)

This notebook runs speaker diarization on the AMI subset audio. It is resumable and writes per-file JSON, RTTM, and a checkpoint index.

Notes:
- No randomness is used, and files are processed in deterministic order.
- Set your Hugging Face token in the configuration cell before running.
- This notebook is diarization-only and does not touch ASR outputs.

In [1]:
from __future__ import annotations

import hashlib
import json
import logging
import os
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable, Optional

import pandas as pd
import torch
from pyannote.audio import Pipeline
try:
    import soundfile as sf
except ImportError:
    sf = None

# ---- Paths (edit here if your folder structure differs) ----
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
RAW_AUDIO_DIR = PROJECT_ROOT / 'data' / 'raw_audio'
OUTPUT_ROOT = PROJECT_ROOT / 'data' / 'diarization_outputs'

JSON_DIR = OUTPUT_ROOT / 'json'
RTTM_DIR = OUTPUT_ROOT / 'rttm'
CSV_DIR = OUTPUT_ROOT / 'csv'
LOGS_DIR = OUTPUT_ROOT / 'logs'
CHECKPOINTS_DIR = OUTPUT_ROOT / 'checkpoints'

for directory in [JSON_DIR, RTTM_DIR, CSV_DIR, LOGS_DIR, CHECKPOINTS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# ---- Configuration ----
AUDIO_EXTS = {'.wav'}
SESSION_PATTERN = r'^(?P<session>[A-Z]{2}\\d{4}[a-d])'

MODEL_ID = 'pyannote/speaker-diarization-3.1'
DEVICE = torch.device('cpu')
DEVICE_STR = str(DEVICE)

# Diarization stability parameters
STABILITY_WINDOW_SEC = 5.0
STABILITY_MAX_NEIGHBORS = 10

# Set your Hugging Face token here or export HF_TOKEN in your environment
HF_TOKEN = os.environ.get('HF_TOKEN') or ''

SKIP_IF_CHECKPOINTED = True
RETRY_FAILED = False

CHECKPOINT_INDEX = CHECKPOINTS_DIR / 'diarization_index.csv'
SUMMARY_CSV = CSV_DIR / 'diarization_summary.csv'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
LOG_PATH = LOGS_DIR / f'diarization_run_{RUN_ID}.log'

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
    handlers=[
        logging.FileHandler(LOG_PATH),
        logging.StreamHandler(),
    ],
)

def load_waveform(path: Path) -> tuple[torch.Tensor, int]:
    if sf is None:
        raise ImportError('soundfile is required for audio loading. Install soundfile and rerun.')
    data, sample_rate = sf.read(str(path), always_2d=True)
    waveform = torch.from_numpy(data.T).float()
    return waveform, int(sample_rate)

/Users/adityagoyal/SRH/Thesis Research/thesis_code/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/adityagoyal/SRH/Thesis Research/thesis_code/.venv/lib/python3.12/site-packages/pyannote/audio/core/io.py:47: UserWarning: 
torchcodec is not installed correctly so built-in audio decoding will fail. Solutions are:
* use audio preloaded in-memory as a {'waveform': (channel, time) torch.Tensor, 'sample_rate': int} dictionary;
* fix torchcodec installation. Error message was:

No module named 'torchcodec'
  warnings.warn(


In [2]:
def iter_files(root: Path, exts: Optional[set[str]] = None) -> Iterable[Path]:
    if not root.exists():
        return []
    for path in sorted(root.rglob('*')):
        if path.is_file() and (exts is None or path.suffix.lower() in exts):
            yield path


def parse_session_id_from_name(name: str) -> Optional[str]:
    match = re.match(SESSION_PATTERN, name)
    return match.group('session') if match else None


def make_file_id(relative_path: str) -> str:
    return hashlib.sha1(relative_path.encode('utf-8')).hexdigest()[:12]


def load_checkpoint(path: Path) -> pd.DataFrame:
    columns = [
        'file_id',
        'relative_path',
        'session_id',
        'status',
        'model_id',
        'device',
        'runtime_seconds',
        'json_path',
        'rttm_path',
        'summary_csv',
        'error',
        'updated_at',
    ]
    if not path.exists():
        return pd.DataFrame(columns=columns)
    df = pd.read_csv(path)
    if df.empty:
        return pd.DataFrame(columns=columns)
    if 'updated_at' in df.columns:
        df = df.sort_values('updated_at').drop_duplicates('file_id', keep='last')
    return df


def save_checkpoint(df: pd.DataFrame, path: Path) -> None:
    df.to_csv(path, index=False)


def write_json(path: Path, payload: dict) -> None:
    with path.open('w', encoding='utf-8') as handle:
        json.dump(payload, handle, ensure_ascii=True, indent=2)


def append_summary_row(row: dict, path: Path) -> None:
    df = pd.DataFrame([row])
    if path.exists():
        df.to_csv(path, mode='a', header=False, index=False)
    else:
        df.to_csv(path, index=False)


def _get_annotation(diarization_output):
    if hasattr(diarization_output, 'itertracks'):
        return diarization_output
    if hasattr(diarization_output, 'speaker_diarization'):
        return diarization_output.speaker_diarization
    if hasattr(diarization_output, 'annotation'):
        return diarization_output.annotation
    if isinstance(diarization_output, dict) and 'annotation' in diarization_output:
        return diarization_output['annotation']
    raise TypeError(f'Unexpected diarization output type: {type(diarization_output)}')


def _build_segment_table(annotation) -> list[dict]:
    segments = []
    for segment, _, label in annotation.itertracks(yield_label=True):
        segments.append({
            'start': float(segment.start),
            'end': float(segment.end),
            'speaker': str(label),
        })
    return sorted(segments, key=lambda seg: (seg['start'], seg['end']))


def _segment_midpoint(segment: dict) -> float:
    return (segment['start'] + segment['end']) / 2.0


def _overlap_fraction(target: dict, segments: list[dict]) -> float:
    duration = max(target['end'] - target['start'], 1e-9)
    overlap = 0.0
    for other in segments:
        if other is target:
            continue
        if other['speaker'] == target['speaker']:
            continue
        inter = min(target['end'], other['end']) - max(target['start'], other['start'])
        if inter > 0:
            overlap += inter
    return min(overlap / duration, 1.0)


def _select_neighbors(target: dict, segments: list[dict], window_sec: float, max_neighbors: int) -> list[dict]:
    center = _segment_midpoint(target)
    candidates = [
        seg for seg in segments
        if seg is not target and abs(_segment_midpoint(seg) - center) <= window_sec
    ]
    candidates.sort(key=lambda seg: abs(_segment_midpoint(seg) - center))
    return candidates[:max_neighbors]


def _seg_consistency(target: dict, neighbors: list[dict]) -> float:
    if not neighbors:
        return 1.0
    same = sum(1 for seg in neighbors if seg['speaker'] == target['speaker'])
    return same / len(neighbors)


def _flip_rate(target: dict, neighbors: list[dict]) -> float:
    window_segments = neighbors + [target]
    window_segments.sort(key=lambda seg: (seg['start'], seg['end']))
    labels = [seg['speaker'] for seg in window_segments]
    if len(labels) <= 1:
        return 0.0
    changes = sum(1 for i in range(1, len(labels)) if labels[i] != labels[i - 1])
    max_changes = len(labels) - 1
    return changes / max_changes


def extract_segments(annotation) -> list[dict]:
    segments = _build_segment_table(annotation)
    out = []
    for segment in segments:
        neighbors = _select_neighbors(
            segment,
            segments,
            window_sec=STABILITY_WINDOW_SEC,
            max_neighbors=STABILITY_MAX_NEIGHBORS,
        )
        seg_consistency = _seg_consistency(segment, neighbors)
        overlap = _overlap_fraction(segment, segments)
        flip_rate = _flip_rate(segment, neighbors)
        diar_stab = (seg_consistency + (1.0 - overlap) + (1.0 - flip_rate)) / 3.0
        out.append({
            'start': segment['start'],
            'end': segment['end'],
            'speaker': segment['speaker'],
            'confidence': float(diar_stab),
            'seg_consistency': float(seg_consistency),
            'overlap': float(overlap),
            'flip_rate': float(flip_rate),
        })
    return out

In [3]:
if not HF_TOKEN:
    raise ValueError('HF_TOKEN is required for pyannote models. Set HF_TOKEN or edit the config cell.')

os.environ['HF_TOKEN'] = HF_TOKEN

if sf is None:
    raise ImportError('soundfile is required to load audio for pyannote. Install soundfile and rerun this cell.')

logging.info('Loading checkpoint index: %s', CHECKPOINT_INDEX)
checkpoint_df = load_checkpoint(CHECKPOINT_INDEX)
checkpoint_df = checkpoint_df.set_index('file_id', drop=False)

audio_files = list(iter_files(RAW_AUDIO_DIR, exts=AUDIO_EXTS))
logging.info('Discovered %s audio files', len(audio_files))
if not audio_files:
    raise FileNotFoundError(f'No audio files found under {RAW_AUDIO_DIR}')

logging.info('Initializing diarization pipeline: %s', MODEL_ID)
try:
    pipeline = Pipeline.from_pretrained(MODEL_ID, token=HF_TOKEN)
except TypeError:
    pipeline = Pipeline.from_pretrained(MODEL_ID)
pipeline.to(DEVICE)

success_count = 0
skip_count = 0
fail_count = 0

for index, audio_path in enumerate(audio_files, start=1):
    relative_path = audio_path.relative_to(PROJECT_ROOT).as_posix()
    session_id = parse_session_id_from_name(audio_path.name)
    file_id = make_file_id(relative_path)

    if file_id in checkpoint_df.index and SKIP_IF_CHECKPOINTED:
        existing_status = str(checkpoint_df.loc[file_id].get('status', '')).lower()
        if existing_status != 'failed' or not RETRY_FAILED:
            logging.info('[%s/%s] Skipping (checkpointed): %s', index, len(audio_files), relative_path)
            skip_count += 1
            continue

    logging.info('[%s/%s] Diarizing: %s', index, len(audio_files), relative_path)
    start_time = time.perf_counter()

    json_path = JSON_DIR / f'{file_id}.json'
    rttm_path = RTTM_DIR / f'{file_id}.rttm'

    try:
        waveform, sample_rate = load_waveform(audio_path)
        diarization_input = {
            'uri': file_id,
            'waveform': waveform,
            'sample_rate': sample_rate,
        }
        diarization = pipeline(diarization_input)
        annotation = _get_annotation(diarization)
        segments = extract_segments(annotation)
        runtime_seconds = time.perf_counter() - start_time

        speakers = sorted({seg['speaker'] for seg in segments})
        summary_row = {
            'file_id': file_id,
            'session_id': session_id,
            'relative_path': relative_path,
            'num_segments': len(segments),
            'num_speakers': len(speakers),
            'model_id': MODEL_ID,
            'device': DEVICE_STR,
            'runtime_seconds': float(runtime_seconds),
            'json_path': str(json_path),
            'rttm_path': str(rttm_path),
        }

        payload = {
            'file_id': file_id,
            'session_id': session_id,
            'relative_path': relative_path,
            'audio_path': str(audio_path),
            'model_id': MODEL_ID,
            'device': DEVICE_STR,
            'runtime_seconds': float(runtime_seconds),
            'segments': segments,
            'created_at': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
            'status': 'success',
        }

        write_json(json_path, payload)
        with rttm_path.open('w', encoding='utf-8') as handle:
            annotation.write_rttm(handle)
        append_summary_row(summary_row, SUMMARY_CSV)

        checkpoint_row = {
            'file_id': file_id,
            'relative_path': relative_path,
            'session_id': session_id,
            'status': 'success',
            'model_id': MODEL_ID,
            'device': DEVICE_STR,
            'runtime_seconds': float(runtime_seconds),
            'json_path': str(json_path),
            'rttm_path': str(rttm_path),
            'summary_csv': str(SUMMARY_CSV),
            'error': None,
            'updated_at': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
        }
        checkpoint_df.loc[file_id] = checkpoint_row
        save_checkpoint(checkpoint_df.reset_index(drop=True), CHECKPOINT_INDEX)

        success_count += 1
        logging.info('[%s/%s] Completed in %.2fs: %s', index, len(audio_files), runtime_seconds, relative_path)
    except Exception as exc:
        runtime_seconds = time.perf_counter() - start_time
        logging.exception('Failed to diarize %s', relative_path)
        logging.info('[%s/%s] Failed after %.2fs: %s', index, len(audio_files), runtime_seconds, relative_path)

        checkpoint_row = {
            'file_id': file_id,
            'relative_path': relative_path,
            'session_id': session_id,
            'status': 'failed',
            'model_id': MODEL_ID,
            'device': DEVICE_STR,
            'runtime_seconds': float(runtime_seconds),
            'json_path': str(json_path),
            'rttm_path': str(rttm_path),
            'summary_csv': str(SUMMARY_CSV),
            'error': repr(exc),
            'updated_at': datetime.now(timezone.utc).isoformat(timespec='seconds').replace('+00:00', 'Z'),
        }
        checkpoint_df.loc[file_id] = checkpoint_row
        save_checkpoint(checkpoint_df.reset_index(drop=True), CHECKPOINT_INDEX)

        fail_count += 1

logging.info('Diarization run complete: %s success, %s skipped, %s failed', success_count, skip_count, fail_count)
summary_df = pd.DataFrame({
    'metric': ['success', 'skipped', 'failed', 'total'],
    'value': [success_count, skip_count, fail_count, len(audio_files)],
})
summary_df

2026-06-01 13:15:47,544 [INFO] Loading checkpoint index: /Users/adityagoyal/SRH/Thesis Research/thesis_code/data/diarization_outputs/checkpoints/diarization_index.csv
2026-06-01 13:15:47,552 [INFO] Discovered 36 audio files
2026-06-01 13:15:47,552 [INFO] Initializing diarization pipeline: pyannote/speaker-diarization-3.1
2026-06-01 13:15:47,811 [INFO] HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-3.1/resolve/main/config.yaml "HTTP/1.1 200 OK"
2026-06-01 13:15:48,912 [INFO] HTTP Request: HEAD https://huggingface.co/pyannote/segmentation-3.0/resolve/main/pytorch_model.bin "HTTP/1.1 302 Found"
2026-06-01 13:15:49,154 [INFO] HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/plda/xvec_transform.npz "HTTP/1.1 302 Found"
2026-06-01 13:15:49,292 [INFO] HTTP Request: HEAD https://huggingface.co/pyannote/speaker-diarization-community-1/resolve/main/plda/plda.npz "HTTP/1.1 302 Found"
2026-06-01 13:15:49,459 [INFO] HTTP Request

,metric,value
0,success,24
1,skipped,12
2,failed,0
3,total,36


## Output Structure
- json/: per-file diarization JSON with segments and metadata
- rttm/: per-file RTTM outputs
- csv/: summary CSV (appended per file)
- checkpoints/diarization_index.csv: master checkpoint table
- logs/: per-run log file